<a href="https://colab.research.google.com/github/sandeshit/ml-togglecorp/blob/pytorch_works/py_lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THIS IS THE IMPLEMENTATION OF PYTORCH LIGHTNING


In [2]:
!pip install pytorch_lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.3/812.3 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.8/868.8 kB 17.4 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-man

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import pytorch_lightning as pl

import numpy as np
import torchmetrics
from torchmetrics import Metric



In [2]:
num_epochs = 50
batch_size = 8
learning_rate = 0.001

In [3]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5,0.5, 0.5),(0.5,0.5,0.5))]
)

In [4]:
train_dataset= torchvision.datasets.CIFAR10(root='./data', train = True, download = True, transform= transform)
test_dataset= torchvision.datasets.CIFAR10(root='./data', train = False, download = True, transform= transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size = batch_size, shuffle = True)

Files already downloaded and verified
Files already downloaded and verified


In [6]:

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

In [7]:
num_classes= len(classes)

In [8]:
#loss_fn = nn.CrossEntropyLoss()

In [8]:
class Convnet(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,16,5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84,10)
        self.loss_fn = nn.CrossEntropyLoss()
        self.accuracy = torchmetrics.Accuracy(task= "multiclass", num_classes= num_classes)
        self.f1_score = torchmetrics.F1Score(task="multiclass", num_classes= num_classes)


    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16*5*5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        loss,scores, y = self.common_step(batch,batch_idx)
        accuracy = self.accuracy(scores,y)
        f1_score = self.f1_score(scores,y)
        self.log_dict({'train_loss': loss, 'train_accuracy': accuracy, 'train_f1_score':f1_score}, on_step = False, on_epoch= True, prog_bar= True)
        return {'loss':loss, 'scores':scores, "y":y}

    def validation_step(self, batch, batch_idx):
        # training_step defines the train loop.
        loss,scores, y = self.common_step(batch,batch_idx)

    def test_step(self, batch, batch_idx):
        # training_step defines the train loop.
        loss,scores, y = self.common_step(batch,batch_idx)

    def common_step(self,batch,batch_idx):

        x, y = batch
        scores = self.forward(x)
        loss = self.loss_fn(scores,y)
        return loss, scores, y

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr = learning_rate)







In [9]:
model= Convnet()

In [10]:
trainer = pl.Trainer(max_epochs=10)
trainer.fit(model, train_loader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
INFO:pytorch_lightning.callbacks.model_summary:
  | Name     | Type               | Params | Mode 
--------------------------------------------------------
0 | conv1    | Conv2d             | 456    | train
1 | pool     | MaxPool2d          | 0      | train
2 | conv2    | Conv2d             | 2.4 K  | train
3 | fc1      | Linear             | 48.1 K | train
4 | fc2      | Linear             | 10.2 K | train
5 | fc3      | Linear             | 850    | train
6 | loss_fn  | CrossEntropyLoss   | 0      | train
7 | accuracy | MulticlassAccuracy | 0      | train
8 | f1_score | Mul

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.
